In [21]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

In [3]:
df = pd.read_csv('data/student_score_data.csv')

In [4]:
df.head()

,Unnamed: 0,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,total_score,average
0,0,female,group B,bachelor's degree,standard,none,72,72,74,218,72.666667
1,1,female,group C,some college,standard,completed,69,90,88,247,82.333333
2,2,female,group B,master's degree,standard,none,90,95,93,278,92.666667
3,3,male,group A,associate's degree,free/reduced,none,47,57,44,148,49.333333
4,4,male,group C,some college,standard,none,76,78,75,229,76.333333


In [5]:
X = df.drop(columns=['average'])

In [6]:
y = df['average']

In [7]:
X.head()

,Unnamed: 0,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,total_score
0,0,female,group B,bachelor's degree,standard,none,72,72,74,218
1,1,female,group C,some college,standard,completed,69,90,88,247
2,2,female,group B,master's degree,standard,none,90,95,93,278
3,3,male,group A,associate's degree,free/reduced,none,47,57,44,148
4,4,male,group C,some college,standard,none,76,78,75,229


### Encoder

In [13]:
from sklearn.compose import ColumnTransformer


categorical_features = X.select_dtypes(include='str').columns
numerical_features = X.select_dtypes(exclude='str').columns

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoding", oh_transformer, categorical_features),
        ("StandardScaler", numeric_transformer, numerical_features),
        
    ]
)
print("Success")


Success


In [14]:
X = preprocessor.fit_transform(X)

In [15]:
X

array([[ 1.        ,  0.        ,  0.        , ...,  0.19399858,
         0.39149181,  0.34357423],
       [ 1.        ,  0.        ,  0.        , ...,  1.42747598,
         1.31326868,  1.0219275 ],
       [ 1.        ,  0.        ,  0.        , ...,  1.77010859,
         1.64247471,  1.74706375],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.12547206,
        -0.20107904, -0.19443008],
       [ 1.        ,  0.        ,  0.        , ...,  0.60515772,
         0.58901542,  0.46053169],
       [ 1.        ,  0.        ,  0.        , ...,  1.15336989,
         1.18158627,  1.06871048]], shape=(1000, 22))

In [16]:
X.shape

(1000, 22)

### Train Test Split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.20, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape


((800, 22), (200, 22), (800,), (200,))

### Create Evaluation Function to give all metrics after model trianing.

In [20]:
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_sqr = r2_score(true, predicted)
    return mae, mse, rmse, r2_sqr

In [35]:
models = {
    'KNeighborsRegressor': KNeighborsRegressor(),
    'DecisionTreeRegressor': DecisionTreeRegressor(),
    'RandomForestRegressor': RandomForestRegressor(),
    'AdaBoostRegressor': AdaBoostRegressor(),
    'GradientBoostingRegressor': GradientBoostingRegressor(),
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'XGBRegressor': XGBRegressor(),
    'CatBoost': CatBoostRegressor(verbose=False)
}

model_list = []
r2_list = []

for i in range(len(models)):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate Train and Test 
    model_train_mae, model_train_mse, model_train_rmse, model_train_r2_sqr = evaluate_model(y_train, y_train_pred)

    model_test_mae, model_test_mse, model_test_rmse, model_test_r2_sqr = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    r2_list.append(model_test_r2_sqr)


    print("Model Performance Training Set")
    print("- Mean Squared Error         : {:.4f}".format(model_train_mse))
    print("- Mean Absolute Error        : {:.4f}".format(model_train_mae))
    print("- Root Mean Squared Error    : {:.4f}".format(model_train_rmse))
    print("- R2 Score                   : {:.4f}".format(model_train_r2_sqr))

    print("Model Performance Test Set")
    print("- Mean Squared Error         : {:.4f}".format(model_test_mse))
    print("- Mean Absolute Error        : {:.4f}".format(model_test_mae))
    print("- Root Mean Squared Error    : {:.4f}".format(model_test_rmse))
    print("- R2 Score                   : {:.4f}".format(model_test_r2_sqr))


    print("-"*40)
    print("\n")

KNeighborsRegressor
Model Performance Training Set
- Mean Squared Error         : 4.5659
- Mean Absolute Error        : 1.6477
- Root Mean Squared Error    : 2.1368
- R2 Score                   : 0.9771
Model Performance Test Set
- Mean Squared Error         : 8.2726
- Mean Absolute Error        : 2.0413
- Root Mean Squared Error    : 2.8762
- R2 Score                   : 0.9614
----------------------------------------


DecisionTreeRegressor
Model Performance Training Set
- Mean Squared Error         : 0.0000
- Mean Absolute Error        : 0.0000
- Root Mean Squared Error    : 0.0000
- R2 Score                   : 1.0000
Model Performance Test Set
- Mean Squared Error         : 0.5217
- Mean Absolute Error        : 0.1117
- Root Mean Squared Error    : 0.7223
- R2 Score                   : 0.9976
----------------------------------------


RandomForestRegressor
Model Performance Training Set
- Mean Squared Error         : 0.0169
- Mean Absolute Error        : 0.0263
- Root Mean Squared

In [36]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=['Model Name', 'R2 Score']).sort_values(by=['R2 Score'], ascending=False)

,Model Name,R2 Score
5,LinearRegression,1.000000
6,Ridge,1.000000
1,DecisionTreeRegressor,0.997566
8,XGBRegressor,0.997402
4,GradientBoostingRegressor,0.996787
2,RandomForestRegressor,0.995679
7,Lasso,0.994801
9,CatBoost,0.993708
3,AdaBoostRegressor,0.989989
0,KNeighborsRegressor,0.961409


[]